# RAG Self-Consistency
LLM의 확률적 특성을 이용해서, 여러번 답변을 생성하고, 그중에 가장 일관된 답변(다수결)을 채택해서 최종응답으로 사용하는 기법이다.

In [1]:
%pip install sentence_transformers scikit-learn -Uqqq

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')


In [4]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content="파리의 상징은 에펠탑이며, 1889년에 세워졌습니다."),  # 에펠탑 기본 정보
        Document(page_content="파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다."),  # 도시 구조 및 대표 박물관
        Document(page_content="파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.")  # 관광 규모 및 랜드마크
    ]
retrieve_vectordb()

[Document(metadata={}, page_content='파리의 상징은 에펠탑이며, 1889년에 세워졌습니다.'),
 Document(metadata={}, page_content='파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다.'),
 Document(metadata={}, page_content='파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.')]

In [5]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', n = 5) # 한 번 요청으로 5개 응답 생성
prompt = ChatPromptTemplate.from_template('''
아래 주어진 문서를 참고해서 사용자의 [질문]에 대한 여행일정을 작성해주세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
- 답변은 **최종추천일정:**으로 시작하세요.
- 일자별 일정은 한문장으로 요약하세요.
- 불필요한 서술은 생략하고, 핵심일정만 나열하세요.
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'

retrieved_docs = retrieve_vectordb(question)
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

response = chain.invoke({'context': context, 'question': question})

print(response)

최종추천일정:
- **방문 시기:** 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.
- **1일차:** 에펠탑을 방문해 1889년 파리 만국박람회와 도시의 역사를 살펴본 뒤 세느강 주변을 산책합니다.
- **2일차:** 루브르 박물관에서 파리의 예술과 역사를 감상하고 주변 시내를 둘러봅니다.
- **3일차:** 개선문을 방문해 파리의 상징적인 도시 경관을 감상하고 주요 관광지를 둘러보며 여행을 마무리합니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', n = 5) # 한 번 요청으로 5개 응답 생성
prompt = ChatPromptTemplate.from_template('''
아래 주어진 문서를 참고해서 사용자의 [질문]에 대한 여행일정을 작성해주세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
- 답변은 **최종추천일정:**으로 시작하세요.
- 일자별 일정은 한문장으로 요약하세요.
- 불필요한 서술은 생략하고, 핵심일정만 나열하세요.
''')

output_parser = StrOutputParser()


question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'

retrieved_docs = retrieve_vectordb(question)
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

messages = prompt.format_prompt(context=context, question=question).to_messages()

response = llm.generate([messages])
print(response)

generations=[[ChatGeneration(text='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 세느강 주변을 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  \n2일차: 에펠탑을 방문해 1889년 건립된 파리의 상징을 둘러보고 세느강변에서 야경을 즐깁니다.  \n3일차: 개선문과 주변 거리를 관광하며 세계적인 관광 도시 파리의 역사와 현재 모습을 체험합니다.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 세느강 주변을 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  \n2일차: 에펠탑을 방문해 1889년 건립된 파리의 상징을 둘러보고 세느강변에서 야경을 즐깁니다.  \n3일차: 개선문과 주변 거리를 관광하며 세계적인 관광 도시 파리의 역사와 현재 모습을 체험합니다.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0602d-6b99-7c61-af3e-73d5f9732c1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1082, 'total_tokens': 1287, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 422}})), ChatGeneration(text='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 파리 역사와 세느강의 도시 발전을 살펴보며 에펠탑과 세느강 주변을 방문하세

In [12]:
from pprint import pprint
pprint(response.generations[0])

[ChatGeneration(text='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 세느강 주변을 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  \n2일차: 에펠탑을 방문해 1889년 건립된 파리의 상징을 둘러보고 세느강변에서 야경을 즐깁니다.  \n3일차: 개선문과 주변 거리를 관광하며 세계적인 관광 도시 파리의 역사와 현재 모습을 체험합니다.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 세느강 주변을 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  \n2일차: 에펠탑을 방문해 1889년 건립된 파리의 상징을 둘러보고 세느강변에서 야경을 즐깁니다.  \n3일차: 개선문과 주변 거리를 관광하며 세계적인 관광 도시 파리의 역사와 현재 모습을 체험합니다.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0602d-6b99-7c61-af3e-73d5f9732c1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1082, 'total_tokens': 1287, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 422}})),
 ChatGeneration(text='최종추천일정:  \n1일차: 봄·가을의 쾌적한 시기에 파리 역사와 세느강의 도시 발전을 살펴보며 에펠탑과 세느강 주변을 방문하세요.  \n2일차: 루

In [14]:
candidates = [gen.text for gen in response.generations[0]]
for cand in candidates:
    print(cand)
    print()

최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 세느강 주변을 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  
2일차: 에펠탑을 방문해 1889년 건립된 파리의 상징을 둘러보고 세느강변에서 야경을 즐깁니다.  
3일차: 개선문과 주변 거리를 관광하며 세계적인 관광 도시 파리의 역사와 현재 모습을 체험합니다.

최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 파리 역사와 세느강의 도시 발전을 살펴보며 에펠탑과 세느강 주변을 방문하세요.  
2일차: 루브르 박물관에서 파리의 예술과 역사를 감상한 뒤 인근 시내를 둘러보세요.  
3일차: 개선문을 방문해 파리의 대표적인 역사·관광 명소를 둘러보며 여행을 마무리하세요.

최종추천일정:  
1일차: 봄·가을 오전에 에펠탑을 방문하고 세느강변을 산책한 뒤, 저녁에는 세느강 야경을 감상합니다.  
2일차: 오전부터 루브르 박물관을 관람하며 파리의 역사와 예술을 살펴보고, 오후에는 튈르리 정원과 시내를 둘러봅니다.  
3일차: 오전에 개선문과 샹젤리제 거리를 방문하고, 오후에는 파리의 주요 역사 지구를 산책하며 여행을 마무리합니다.

최종추천일정:  
1일차: 봄·가을의 오전에 세느강을 따라 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  
2일차: 오후에 1889년 건립된 에펠탑을 방문해 파리의 상징과 세느강의 전경을 즐깁니다.  
3일차: 오전에는 개선문을 둘러보고 주변 시내를 관광하며 세계적인 관광 도시 파리의 매력을 만끽합니다.

최종추천일정:  
- **1일차:** 봄·가을의 쾌적한 시기에 에펠탑(1889년 건립)을 방문하고 세느강변을 산책합니다.  
- **2일차:** 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 주변 명소를 둘러봅니다.  
- **3일차:** 개선문을 방문해 파리의 대표적인 도시 경관을 감상하고 주요 관광지를 여유롭게 둘러봅니다.



### n개의 응답을 하나로 추출하기

In [ ]:
from langchain_core.output_parsers import BaseOutputParser # 출력 결과를 원하는 형식으로 변환용 기본 Parser
from sentence_transformers import SentenceTransformer # 임베딩 모델
from pydantic import Field # 클래스 속성 정의/검증용
from sklearn.cluster import KMeans # 클러스터링 모델
from collections import Counter 
import numpy as np

class RobustSelfConsistencyParser(BaseOutputParser):
    n_clusters: int = Field(default=2)
    encoder: object = Field(default=SentenceTransformer('all-MiniLM-L6-v2'))

    def parse(self, generations: list[str]) -> str:
        # 1. 임베딩
        embeddings = self.encoder.encode(generations)

        print(embeddings.shape) # (후보 답변 갯수, 임베딩 벡터 차원)
        # 2. 클러스터링(KMeans)
        kmeans = KMeans(n_clusters=self.n_clusters, random_state=42)
        kmeans.fit(embeddings) # 클러스터링 실행
        print(kmeans.labels_)  # 각 답변이 어느 라벨에 속해 있는지 확인
        # 3. 다수결 투표
        counts = Counter(kmeans.labels_)
        target_label = max(counts, key = counts.get) # 가장 많은 후보 답변이 속한 클러스터
        # 선택된 클러스터에 속한 후보 답변 인덱스만 추출
        target_indeces = np.where(kmeans.labels_ == target_label)[0]
        print(target_label)
        print(target_indeces)
        # 4. 대표 답변 선택 (중심점에 가장 가까운 후보)
        target_centroid = kmeans.cluster_centers_[target_label]
        # 후보 답변들과 중심점 사이의 거리를 계산
        distances = np.linalg.norm(embeddings[target_indeces] - target_centroid, axis=1)
        representive_idx = np.argmin(distances) # 중심점과 가장 가까운 답변 인덱스
        return generations[target_indeces[representive_idx]] # 클러스터 중 가장 대표답변 반환

parser = RobustSelfConsistencyParser()
final_answer = parser.parse(candidates) # 후보 중 대표 답변 선택
print(f"최종 답변 : {final_answer}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(5, 384)
[1 0 0 1 1]
1
[0 3 4]
최종 답변 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 세느강 주변을 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  
2일차: 에펠탑을 방문해 1889년 건립된 파리의 상징을 둘러보고 세느강변에서 야경을 즐깁니다.  
3일차: 개선문과 주변 거리를 관광하며 세계적인 관광 도시 파리의 역사와 현재 모습을 체험합니다.


In [ ]:
def travel_planner(question, verbose=False):
    retrieved_docs = retrieve_vectordb(question)
    context = '\n\n'.join([doc.page_content for doc in retrieved_docs])
    messages = prompt.format_prompt(context=context, question=question).to_messages()
    response = llm.generate([messages])
    candidates = [gen.text for gen in response.generation[0]]

    if verbose:
        for i, cand in enumerate(candidates):
            print(f"{i+1} : {cand}")
            print()
    parser = RobustSelfConsistencyParser()
    return parser.parse(candidates)

question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'

response = travel_planner(question, verbose = True)

print(response)

1 : 최종추천일정:

- **1일차:** 세느강변을 산책하며 파리의 역사적 중심지를 둘러보고 루브르 박물관을 관람합니다.
- **2일차:** 1889년에 세워진 에펠탑을 방문한 뒤 주변 세느강 경관을 감상합니다.
- **3일차:** 개선문과 샹젤리제 거리를 관광하며 파리의 상징적인 명소를 마무리로 둘러봅니다.
- **추천 시기:** 날씨가 쾌적하고 관광하기 좋은 봄(4~5월) 또는 가을(9~10월)에 방문하세요.

2 : 최종추천일정:  
- **1일차(봄·가을 추천):** 세느강 산책으로 파리의 도시 역사를 살펴본 뒤 루브르 박물관과 주변 역사 지구를 방문합니다.  
- **2일차:** 에펠탑을 관람하고 세느강 주변에서 파리의 상징적인 풍경을 즐깁니다.  
- **3일차:** 개선문과 샹젤리제 거리를 둘러보며 근현대 파리의 역사와 관광 명소를 체험합니다.

3 : 최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  
- 1일차: 세느강을 따라 산책한 뒤 루브르 박물관을 관람하며 파리의 역사와 문화를 둘러봅니다.  
- 2일차: 1889년에 세워진 에펠탑을 방문하고 주변에서 파리 전경을 감상합니다.  
- 3일차: 개선문과 샹젤리제 거리를 둘러보며 파리의 대표적인 관광지를 마무리합니다.

4 : 최종추천일정:  
- **방문 시기:** 쾌적한 날씨와 비교적 여유로운 관광이 가능한 봄(4~5월) 또는 가을(9~10월)을 추천합니다.  
- **1일차:** 세느강 산책으로 파리의 역사적 도시 풍경을 감상한 뒤 루브르 박물관과 주변 구시가지를 둘러봅니다.  
- **2일차:** 에펠탑을 방문해 1889년 건립된 파리의 상징을 감상하고, 샹드마르스와 세느강변을 산책합니다.  
- **3일차:** 개선문과 샹젤리제 거리를 관광하며 파리의 근현대 역사와 대표적인 도시 경관을 감상합니다.

5 : 최종추천일정:
- 1일차: 세느강 산책과 루브르 박물관을 둘러보며 파리의 역사와 예술을 감상하세요.
- 2일차: 1889년에 세워진 에펠탑